<a href="https://colab.research.google.com/github/Ca3ra/AIchatbot/blob/phase-3/AI3_Week3_retrieval%20ver1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BOI Investors' Guidebook RAG Chatbot: Week 3 (Retrieval)

Week 3 turns the 1,281 chunks from Week 2 (`chunks.jsonl`) into a searchable index and measures how well it finds the right page.

1. Write the development (dev) questions, each with a gold page, and check every gold page against the guidebook text.
2. Embed the chunks with MiniLM (English baseline) and two multilingual models, and save each FAISS index and the chunk metadata to Drive.
3. Measure hit@k, Precision@k, Recall@k and MRR@k for k = 1, 3, 5, separately for English and Tagalog/Taglish questions.
4. Add BM25 and compare dense only with hybrid (dense + BM25).
5. Check table chunks and test whether chunk size 1000 holds up.

**Definition used everywhere:** a retrieved chunk is *relevant* when its `pdf_pages` contains the question's gold page.

**Run order:** Cells run top to bottom. The embedding cell takes a few minutes the first time and is cached on Drive afterwards.

## Cell 1: Setup

In [1]:
!pip -q install faiss-cpu rank-bm25 sentence-transformers
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2: Config and load the Week 2 output

In [2]:
import json, re, csv, time, random, hashlib
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

BASE = Path("/content/drive/MyDrive/AI3-Project")
PROCESSED = BASE / "data/processed"
INDEX_DIR = BASE / "index"
LOGS = BASE / "logs"
for p in (INDEX_DIR, LOGS):
    p.mkdir(parents=True, exist_ok=True)

K_VALUES = (1, 3, 5)
TOP_K_MAX = max(K_VALUES)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

chunks = read_jsonl(PROCESSED / "chunks.jsonl")
pages = read_jsonl(PROCESSED / "pages_clean.jsonl")
page_text = {r["pdf_page"]: " ".join(r["clean_text"].split()) for r in pages}   # line wraps -> single spaces

assert len({c["chunk_id"] for c in chunks}) == len(chunks), "duplicate chunk ids"
print(f"Loaded {len(chunks)} chunks and {len(pages)} pages")

# Fingerprint of the chunk set: cached indexes are only reused when the chunks have not changed
CHUNK_FP = hashlib.md5("".join(c["chunk_id"] + c["text"] for c in chunks).encode("utf-8")).hexdigest()
print("Chunk fingerprint:", CHUNK_FP[:8])

Loaded 1281 chunks and 502 pages
Chunk fingerprint: 6d02fd7b


## Cell 3: Development questions
20 dev questions written from the guidebook text. Every answerable question has a `gold_page` (PDF page) and a `must_contain` phrase, and the next cell checks that phrase against the page, so each expected answer can be traced to a page.

| Type | What it tests |
|---|---|
| `direct` | the question uses the guidebook's own words |
| `paraphrase` | the same fact in different words |
| `acronym` | the question is mostly acronyms (FAOM, CIR, BE) |
| `lookalike` | similar pages or tables exist (several fee tables, several FDAC procedures, PWP vs SWP) |
| `unanswerable` | the guidebook has no answer; used to look at retrieval scores for the refusal rule in Week 4 |

`lang` is `en` or `tl` (Tagalog or Taglish). These 20 are dev questions used for tuning; the held-out test questions are written later and run once.

In [3]:
DEV_QUESTIONS = [
    # --- carried over from Week 1 (the first five dev questions) ---
    {"id": "D01", "lang": "en", "qtype": "direct", "gold_page": 465, "must_contain": "3,000",
     "q": "What is the BOI registration fee for a micro project?",
     "expected": "PhP 3,000"},
    {"id": "D02", "lang": "en", "qtype": "direct", "gold_page": 39, "must_contain": "at least 35 years old",
     "q": "What is the minimum age of a principal applicant for the Special Retiree's Resident Visa (SRRV)?",
     "expected": "At least 35 years old"},
    {"id": "D03", "lang": "en", "qtype": "direct", "gold_page": 68, "must_contain": "BIR Form No. 1902",
     "q": "Which BIR form is needed to apply online for a TIN for a local employee, and by when must the documents be submitted?",
     "expected": "BIR Form No. 1902 through eREG, before the 10th day of the following month"},
    {"id": "D04", "lang": "en", "qtype": "lookalike", "gold_page": 20, "must_contain": "three to six months",
     "q": "How long is the work covered by a Special Work Permit (SWP) - Commercial?",
     "expected": "Gainful employment of three to six months"},
    {"id": "D05", "lang": "en", "qtype": "direct", "gold_page": 454, "must_contain": "DOLE Regional Offices",
     "q": "Which office issues the Certificate of Registration of Job Contractors and Sub-contractors?",
     "expected": "DOLE Regional Offices (Articles 106 to 109 of the Labor Code)"},
    # --- new in Week 3 ---
    {"id": "D06", "lang": "en", "qtype": "paraphrase", "gold_page": 13, "must_contain": "Department of Labor and Employment",
     "q": "Which government department must approve a foreigner's permit before he or she can be employed in the Philippines?",
     "expected": "Department of Labor and Employment (DOLE), through the Alien Employment Permit"},
    {"id": "D07", "lang": "en", "qtype": "acronym", "gold_page": 115, "must_contain": "First Action on the Merits",
     "q": "What does FAOM stand for?",
     "expected": "First Action on the Merits (FAOM) Report"},
    {"id": "D08", "lang": "en", "qtype": "acronym", "gold_page": 101, "must_contain": "Certificate of Inward Remittance",
     "q": "What is the CIR for an FX inward remittance?",
     "expected": "Certificate of Inward Remittance of FX through an AAB, as proof of funding"},
    {"id": "D09", "lang": "en", "qtype": "lookalike", "gold_page": 119, "must_contain": "1,800",
     "q": "What is the priority claim fee for a big entity?",
     "expected": "Big Entity PhP 1,800 (Small Entity PhP 860)"},
    {"id": "D10", "lang": "en", "qtype": "lookalike", "gold_page": 115, "must_contain": "4,200",
     "q": "What is the substantive examination fee for a big entity?",
     "expected": "Big Entity PhP 4,200 (Small Entity PhP 2,010)"},
    {"id": "D11", "lang": "en", "qtype": "paraphrase", "gold_page": 126, "must_contain": "Industrial Design in 5 days",
     "q": "How fast is the express registration of an industrial design?",
     "expected": "\"Industrial Design in 5 days\" (Utility Model in 2 Months)"},
    {"id": "D12", "lang": "en", "qtype": "lookalike", "gold_page": 380, "must_contain": "Thursdays",
     "q": "On which day can applicants submit requirements to FDAC for pre-assessment?",
     "expected": "Thursdays, 8AM to 12NN (except holidays)"},
    {"id": "D13", "lang": "en", "qtype": "acronym", "gold_page": 300, "must_contain": "12 days",
     "q": "How many days does the desktop review of a BE inspection application take?",
     "expected": "12 days"},
    {"id": "D14", "lang": "en", "qtype": "direct", "gold_page": 80, "must_contain": "PhiFintech Innovation Office",
     "q": "Which SEC office gives clearance for FinTech-related business activities?",
     "expected": "PhiFintech Innovation Office (PIO)"},
    # --- Tagalog and Taglish ---
    {"id": "D15", "lang": "tl", "qtype": "paraphrase", "gold_page": 13, "must_contain": "Department of Labor and Employment",
     "q": "Anong ahensya ng gobyerno ang nagbibigay ng Alien Employment Permit (AEP)?",
     "expected": "Department of Labor and Employment (DOLE)"},
    {"id": "D16", "lang": "tl", "qtype": "direct", "gold_page": 141, "must_contain": "contact_us@dar.gov.ph",
     "q": "Ano ang email address ng DAR?",
     "expected": "contact_us@dar.gov.ph"},
    {"id": "D17", "lang": "tl", "qtype": "lookalike", "gold_page": 115, "must_contain": "4,200",
     "q": "Magkano ang Substantive Examination Fee para sa Big Entity?",
     "expected": "Big Entity PhP 4,200 (same fact as D10, asked in Taglish)"},
    {"id": "D18", "lang": "tl", "qtype": "direct", "gold_page": 500, "must_contain": "FONDEVILLA",
     "q": "Sino ang Executive Director ng BOI Investments Assistance Center?",
     "expected": "Atty. Bobby G. Fondevilla"},
    # --- unanswerable (no gold page) ---
    {"id": "D19", "lang": "en", "qtype": "unanswerable", "gold_page": None, "must_contain": None,
     "q": "What is the visa fee for Philippine citizens visiting Japan?",
     "expected": "Not in the guidebook: the chatbot should say it does not know"},
    {"id": "D20", "lang": "tl", "qtype": "unanswerable", "gold_page": None, "must_contain": None,
     "q": "Magkano ang presyo ng bigas ngayon sa palengke?",
     "expected": "Not in the guidebook: the chatbot should say it does not know"},
]

# 1) every expected answer must be on its gold page (case-insensitive, line wraps ignored)
for d in DEV_QUESTIONS:
    if d["gold_page"] is None:
        continue
    assert d["gold_page"] in page_text, f"{d['id']}: PDF page {d['gold_page']} not found"
    assert d["must_contain"].lower() in page_text[d["gold_page"]].lower(), \
        f"{d['id']}: '{d['must_contain']}' not found on PDF page {d['gold_page']}"

# 2) relevant chunks per question = chunks whose pdf_pages contains the gold page
answerable = [d for d in DEV_QUESTIONS if d["gold_page"] is not None]
unanswerable = [d for d in DEV_QUESTIONS if d["gold_page"] is None]
relevant = {d["id"]: {i for i, c in enumerate(chunks) if d["gold_page"] in c["pdf_pages"]} for d in answerable}
empty = [qid for qid, s in relevant.items() if not s]
assert not empty, f"no chunk covers the gold page for: {empty}"

print(f"{len(DEV_QUESTIONS)} dev questions: {len(answerable)} answerable, {len(unanswerable)} unanswerable")
print("By language:", dict(Counter(d["lang"] for d in DEV_QUESTIONS)))
print("By type:    ", dict(Counter(d["qtype"] for d in DEV_QUESTIONS)))
print("Relevant chunks per question:", {qid: len(s) for qid, s in relevant.items()})
print("All expected answers found on their gold pages.")

20 dev questions: 18 answerable, 2 unanswerable
By language: {'en': 15, 'tl': 5}
By type:     {'direct': 7, 'lookalike': 5, 'paraphrase': 3, 'acronym': 3, 'unanswerable': 2}
Relevant chunks per question: {'D01': 1, 'D02': 4, 'D03': 3, 'D04': 4, 'D05': 4, 'D06': 4, 'D07': 2, 'D08': 6, 'D09': 3, 'D10': 2, 'D11': 4, 'D12': 4, 'D13': 4, 'D14': 5, 'D15': 4, 'D16': 1, 'D17': 2, 'D18': 2}
All expected answers found on their gold pages.


## Cell 4: Embedding models
| Key | Model | Why |
|---|---|---|
| `minilm` | all-MiniLM-L6-v2 | English baseline from Week 2 |
| `e5_base` | multilingual-e5-base | small multilingual model; needs the `query: ` and `passage: ` prefixes |
| `bge_m3` | BGE-M3 | larger multilingual model with a long context; no prefixes |

The guidebook stays in English. The multilingual models are there so that a Tagalog question lands near the English chunk that answers it. To skip a model, remove its key from `MODELS_TO_RUN`. Check the exact model names on Hugging Face before you cite them.

In [4]:
import faiss
from sentence_transformers import SentenceTransformer

MODELS = {
    "minilm": {"hf_id": "sentence-transformers/all-MiniLM-L6-v2", "q_prefix": "", "p_prefix": "", "max_len": None},
    "e5_base": {"hf_id": "intfloat/multilingual-e5-base", "q_prefix": "query: ", "p_prefix": "passage: ", "max_len": 512},
    "bge_m3": {"hf_id": "BAAI/bge-m3", "q_prefix": "", "p_prefix": "", "max_len": 512},   # window is 8k tokens; chunks are short, so 512 is enough
}
MODELS_TO_RUN = ["minilm", "e5_base", "bge_m3"]
FORCE_REBUILD = False          # True = ignore cached indexes on Drive

def load_model(key):
    cfg = MODELS[key]
    m = SentenceTransformer(cfg["hf_id"])           # uses the GPU automatically when there is one
    if cfg["max_len"]:
        m.max_seq_length = min(m.max_seq_length, cfg["max_len"])
    return m

def passage_texts(key, chs):
    return [MODELS[key]["p_prefix"] + c["text"] for c in chs]

def token_stats(model, texts):
    lens = [len(model.tokenizer.encode(t, add_special_tokens=True)) for t in texts]
    limit = model.max_seq_length
    over = sum(1 for n in lens if n > limit)
    return {"max_seq_length": limit, "avg_tokens": round(sum(lens) / len(lens), 1), "max_tokens": max(lens),
            "over_limit": over, "over_limit_pct": round(100 * over / len(lens), 1)}

models, indexes, token_rows, embed_seconds = {}, {}, [], {}
for key in MODELS_TO_RUN:
    m = load_model(key)
    models[key] = m
    token_rows.append({"model": key, **token_stats(m, passage_texts(key, chunks))})

    idx_path = INDEX_DIR / f"{key}_{CHUNK_FP[:8]}.faiss"
    if idx_path.exists() and not FORCE_REBUILD:
        indexes[key] = faiss.read_index(str(idx_path))
        embed_seconds[key] = None
        print(f"{key}: loaded cached index {idx_path.name}")
    else:
        t0 = time.time()
        emb = m.encode(passage_texts(key, chunks), batch_size=32, normalize_embeddings=True,
                       show_progress_bar=True, convert_to_numpy=True).astype("float32")
        index = faiss.IndexFlatIP(emb.shape[1])        # exact search; vectors are normalised, so inner product = cosine
        index.add(emb)
        faiss.write_index(index, str(idx_path))
        indexes[key] = index
        embed_seconds[key] = round(time.time() - t0, 1)
        print(f"{key}: embedded {len(chunks)} chunks in {embed_seconds[key]}s, saved {idx_path.name}")
    assert indexes[key].ntotal == len(chunks), f"{key}: index size differs from the chunk count"

# Chunk metadata in index order (row i of every index = chunks[i])
with open(INDEX_DIR / "chunk_meta.json", "w", encoding="utf-8") as f:
    json.dump({"fingerprint": CHUNK_FP, "n_chunks": len(chunks), "models": {k: MODELS[k] for k in MODELS_TO_RUN},
               "chunks": chunks}, f, ensure_ascii=False)
print("Saved chunk_meta.json and", len(indexes), "index file(s) to", INDEX_DIR)

print("\nDoes every chunk fit the model's window? (text beyond max_seq_length is not embedded)")
display(pd.DataFrame(token_rows))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (37 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

bge_m3: loaded cached index bge_m3_6d02fd7b.faiss
Saved chunk_meta.json and 3 index file(s) to /content/drive/MyDrive/AI3-Project/index

Does every chunk fit the model's window? (text beyond max_seq_length is not embedded)


,model,max_seq_length,avg_tokens,max_tokens,over_limit,over_limit_pct
0,minilm,256,182.5,378,36,2.8
1,e5_base,512,217.1,393,0,0.0
2,bge_m3,512,215.1,391,0,0.0


## Cell 5: Retrieval functions and metrics
- **hit@k:** 1 if at least one of the top k chunks is relevant.
- **Precision@k:** relevant chunks in the top k, divided by k.
- **Recall@k:** relevant chunks in the top k, divided by all relevant chunks for that question. A page is often split over several chunks, so Recall@k stays low even when retrieval works; read it together with hit@k.
- **MRR@k:** 1 / rank of the first relevant chunk (0 if none in the top k).

Metrics use the answerable questions only. The unanswerable ones are used in Cell 9.

In [5]:
def dense_search(key, query, n=TOP_K_MAX):
    q = models[key].encode([MODELS[key]["q_prefix"] + query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    scores, ids = indexes[key].search(q, n)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]

def metrics_for(ranked, rel, k):
    hits = [1 if i in rel else 0 for i in ranked[:k]]
    n_rel = sum(hits)
    return {"hit": int(n_rel > 0), "precision": n_rel / k, "recall": n_rel / len(rel) if rel else 0.0,     # a chunk set may not cover the gold page at all
            "mrr": next((1 / (r + 1) for r, h in enumerate(hits) if h), 0.0)}

def evaluate(search_fn, label, questions=None):
    # search_fn(query, n) -> [(chunk_index, score), ...] best first
    rows = []
    for d in (questions or answerable):
        ranked = [i for i, _ in search_fn(d["q"], TOP_K_MAX)]
        for k in K_VALUES:
            rows.append({"system": label, "id": d["id"], "lang": d["lang"], "qtype": d["qtype"], "k": k,
                         **metrics_for(ranked, relevant[d["id"]], k)})
    return pd.DataFrame(rows)

def summarize(df):
    both = pd.concat([df, df.assign(lang="all")])
    s = both.groupby(["system", "lang", "k"])[["hit", "precision", "recall", "mrr"]].mean().round(3).reset_index()
    s["n_questions"] = both.groupby(["system", "lang", "k"])["id"].nunique().values
    return s

def hit_table(summary):
    t = summary.pivot_table(index=["system", "lang"], columns="k", values="hit")
    t.columns = [f"hit@{k}" for k in t.columns]
    return t

all_results = []     # every evaluation is appended here and saved in Cell 11

## Cell 6: Dense retrieval, each model

In [6]:
dense_results = {}
for key in MODELS_TO_RUN:
    df = evaluate(lambda q, n, key=key: dense_search(key, q, n), f"dense:{key}")
    dense_results[key] = df
    all_results.append(df)

dense_summary = summarize(pd.concat(dense_results.values()))
display(Markdown("### hit@k by model and language"))
display(hit_table(dense_summary))
display(Markdown("### Full metrics (English, Tagalog and all questions)"))
display(dense_summary)

### hit@k by model and language

hit@1  hit@3  hit@5
system        lang                     
dense:bge_m3  all   0.833  0.944  1.000
              en    0.786  0.929  1.000
              tl    1.000  1.000  1.000
dense:e5_base all   0.667  0.889  0.944
              en    0.571  0.857  0.929
              tl    1.000  1.000  1.000
dense:minilm  all   0.556  0.722  0.833
              en    0.500  0.714  0.786
              tl    0.750  0.750  1.000

### Full metrics (English, Tagalog and all questions)

,system,lang,k,hit,precision,recall,mrr,n_questions
0,dense:bge_m3,all,1,0.833,0.833,0.338,0.833,18
1,dense:bge_m3,all,3,0.944,0.500,0.544,0.889,18
2,dense:bge_m3,all,5,1.000,0.400,0.684,0.900,18
3,dense:bge_m3,en,1,0.786,0.786,0.274,0.786,14
4,dense:bge_m3,en,3,0.929,0.476,0.449,0.857,14
5,dense:bge_m3,en,5,1.000,0.414,0.630,0.871,14
6,dense:bge_m3,tl,1,1.000,1.000,0.562,1.000,4
7,dense:bge_m3,tl,3,1.000,0.583,0.875,1.000,4
8,dense:bge_m3,tl,5,1.000,0.350,0.875,1.000,4
9,dense:e5_base,all,1,0.667,0.667,0.245,0.667,18


## Cell 7: BM25 and hybrid retrieval
BM25 matches exact words, so it should help with acronyms and names (AEP, FAOM, CIR). It should help less for Tagalog questions on English text, because most of the words will not match. Hybrid search merges the dense ranking and the BM25 ranking with reciprocal rank fusion (RRF): each chunk scores `1 / (60 + rank)` in each list and the scores are added.

In [7]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())      # keeps acronyms and numbers such as "4" and "200"

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

def bm25_search(query, n=TOP_K_MAX):
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(-scores)[:n]
    return [(int(i), float(scores[i])) for i in order]

HYBRID_POOL = 20       # candidates taken from each list before fusion
RRF_K = 60

def rrf_fuse(ranked_lists, k=RRF_K):
    fused = defaultdict(float)
    for lst in ranked_lists:
        for rank, (i, _) in enumerate(lst):
            fused[i] += 1.0 / (k + rank + 1)
    return sorted(fused.items(), key=lambda x: -x[1])

def hybrid_search(key, query, n=TOP_K_MAX):
    return rrf_fuse([dense_search(key, query, HYBRID_POOL), bm25_search(query, HYBRID_POOL)])[:n]

bm25_df = evaluate(bm25_search, "bm25")
hybrid_dfs = {key: evaluate(lambda q, n, key=key: hybrid_search(key, q, n), f"hybrid:{key}") for key in MODELS_TO_RUN}
all_results += [bm25_df] + list(hybrid_dfs.values())

compare = summarize(pd.concat([bm25_df] + list(dense_results.values()) + list(hybrid_dfs.values())))
display(Markdown("### hit@k: BM25 vs dense vs hybrid, by language"))
display(hit_table(compare))

### hit@k: BM25 vs dense vs hybrid, by language

hit@1  hit@3  hit@5
system         lang                     
bm25           all   0.611  0.722  0.778
               en    0.571  0.714  0.786
               tl    0.750  0.750  0.750
dense:bge_m3   all   0.833  0.944  1.000
               en    0.786  0.929  1.000
               tl    1.000  1.000  1.000
dense:e5_base  all   0.667  0.889  0.944
               en    0.571  0.857  0.929
               tl    1.000  1.000  1.000
dense:minilm   all   0.556  0.722  0.833
               en    0.500  0.714  0.786
               tl    0.750  0.750  1.000
hybrid:bge_m3  all   0.778  1.000  1.000
               en    0.714  1.000  1.000
               tl    1.000  1.000  1.000
hybrid:e5_base all   0.722  0.944  0.944
               en    0.643  0.929  0.929
               tl    1.000  1.000  1.000
hybrid:minilm  all   0.667  0.833  0.944
               en    0.571  0.786  0.929
               tl    1.000  1.000  1.000

## Cell 8: Which setup is best, and where does it fail?

In [8]:
# Best system = highest hit@5 over all questions; ties are broken by MRR@5
cmp5 = compare[(compare["lang"] == "all") & (compare["k"] == TOP_K_MAX)].sort_values(["hit", "mrr"], ascending=False)
display(cmp5[["system", "hit", "precision", "recall", "mrr"]].reset_index(drop=True))
BEST = cmp5.iloc[0]["system"]
print("Best system on these dev questions:", BEST)

def _search_fn(system):
    kind, _, key = system.partition(":")
    if kind == "bm25":
        return bm25_search
    if kind == "dense":
        return lambda q, n: dense_search(key, q, n)
    return lambda q, n: hybrid_search(key, q, n)

fn = _search_fn(BEST)
misses = []
for d in answerable:
    ranked = fn(d["q"], TOP_K_MAX)
    got = [i for i, _ in ranked]
    if not (set(got) & relevant[d["id"]]):
        misses.append({"id": d["id"], "lang": d["lang"], "qtype": d["qtype"], "gold_page": d["gold_page"],
                       "question": d["q"][:70], "top1_pages": chunks[got[0]]["pdf_pages"], "top1_procedure": chunks[got[0]]["procedure"][:50]})
print(f"\nQuestions with no relevant chunk in the top {TOP_K_MAX} ({len(misses)} of {len(answerable)}):")
display(pd.DataFrame(misses) if misses else "none")

# Same comparison by question type, for the best system
by_type = pd.concat([df for df in all_results if df["system"].iloc[0] == BEST])
display(Markdown(f"### {BEST}: hit@{TOP_K_MAX} by question type"))
display(by_type[by_type["k"] == TOP_K_MAX].groupby("qtype")["hit"].agg(["mean", "count"]).round(2))

,system,hit,precision,recall,mrr
0,dense:bge_m3,1.000,0.400,0.684,0.900
1,hybrid:bge_m3,1.000,0.344,0.613,0.889
2,hybrid:e5_base,0.944,0.378,0.670,0.824
3,dense:e5_base,0.944,0.356,0.596,0.792
4,hybrid:minilm,0.944,0.344,0.585,0.756
5,dense:minilm,0.833,0.289,0.483,0.655
6,bm25,0.778,0.300,0.506,0.681


Best system on these dev questions: dense:bge_m3

Questions with no relevant chunk in the top 5 (0 of 18):


'none'

### dense:bge_m3: hit@5 by question type

,mean,count
qtype,,
acronym,1.0,3
direct,1.0,7
lookalike,1.0,5
paraphrase,1.0,3


## Cell 9: Unanswerable questions (scores for the refusal rule)
The chatbot has to refuse when the answer is not in the guidebook. Here we only look at the top-1 dense similarity score of answerable and unanswerable questions. If the two groups separate, a score threshold can support the refusal rule in Week 4; with only a few unanswerable questions this is a first look, not a result.

In [9]:
rows = []
for key in MODELS_TO_RUN:
    ans = [dense_search(key, d["q"], 1)[0][1] for d in answerable]
    una = {d["id"]: round(dense_search(key, d["q"], 1)[0][1], 3) for d in unanswerable}
    rows.append({"model": key, "answerable top-1 min": round(min(ans), 3), "answerable top-1 mean": round(float(np.mean(ans)), 3),
                 **{f"unanswerable {k}": v for k, v in una.items()}})
display(pd.DataFrame(rows))
print("Scores are cosine similarities from each model, so compare them within a row, not across models.")

,model,answerable top-1 min,answerable top-1 mean,unanswerable D19,unanswerable D20
0,minilm,0.327,0.615,0.535,0.361
1,e5_base,0.780,0.862,0.832,0.768
2,bge_m3,0.498,0.668,0.570,0.433


Scores are cosine similarities from each model, so compare them within a row, not across models.


## Cell 10: Table chunks and chunk size
**Table chunks.** A fee or processing-time cell is only useful if the chunk also carries its column label. Week 2 does not repeat a table's header row in every chunk, so later chunks of a table may show numbers without labels. The cell below measures how often that happens and prints a few examples to read.

In [10]:
HEADER_WORDS = re.compile(r"\b(fees?|processing time|agency actions?|client|step|requirements?)\b", re.IGNORECASE)
table_chunks = [c for c in chunks if c.get("has_table")]
money_or_days = [c for c in table_chunks if re.search(r"₱|\bPhP\b|\bdays?\b", c["text"])]
labelled = [c for c in money_or_days if HEADER_WORDS.search(c["text"][c["breadcrumb_len"]:])]
print(f"{len(table_chunks)} table chunks; {len(money_or_days)} contain a fee or a number of days;")
print(f"{len(labelled)} of those ({100 * len(labelled) / max(1, len(money_or_days)):.0f}%) also contain a column label such as FEES, PROCESSING TIME or STEP.")

rng = random.Random(SEED)
unlabelled = [c for c in money_or_days if c not in labelled]
print(f"\n{len(unlabelled)} fee/days chunks without a column label. Five to read:")
for c in rng.sample(unlabelled, k=min(5, len(unlabelled))):
    print(f"\n--- {c['chunk_id']} pages={c['pdf_pages']} ---\n{c['text'][:400]}")

653 table chunks; 369 contain a fee or a number of days;
318 of those (86%) also contain a column label such as FEES, PROCESSING TIME or STEP.

51 fee/days chunks without a column label. Five to read:

--- w2_c0860 pages=[345] ---
[Center for Drug Regulation Research > Certificate of Product Registration (CPR) of Pharmaceutical Products (Initial...] Reviews the evaluated application bearing the recommendation of the Junior Evaluator | 12 days Prepares the final output document (CPR/LOD), affixes initial, and forwards it to the senior evaluator (FDRO III) If with post-approval commitment/s, prepares a letter, signs, and forw

--- w2_c0373 pages=[156, 157] ---
[Energy Resources Development Bureau > Acknowledgment of Administrative Subcontracts, Letters of Registrations, Defi...] Technical Evaluation | 8 days Evaluate subcontracts in accordance to the submitted Work Program and Budget for the Calendar Year for Petroleum Operations related subcontracts and/or Administrative subcontracts No

**Chunk size.** Week 2 chose 1000 characters with a 150-character overlap, marked provisional. Cell W2-6b saves the four sizes that were compared (600, 800, 1000, 1500) to `data/processed/experiments/`. Here each size is embedded with the best dense model and scored with the same questions and the same relevance rule. The overlap is 150 characters at every size (25% of 600, 10% of 1500), so the comparison is not perfectly fair; treat small differences with caution.

In [11]:
EXP_DIR = PROCESSED / "experiments"
BEST_DENSE = max(MODELS_TO_RUN, key=lambda k: dense_summary[(dense_summary["system"] == f"dense:{k}") & (dense_summary["lang"] == "all")
                                                         & (dense_summary["k"] == TOP_K_MAX)]["hit"].iloc[0])

def evaluate_chunk_set(key, chs):
    # embed this chunk set and score it with the answerable dev questions
    emb = models[key].encode(passage_texts(key, chs), batch_size=32, normalize_embeddings=True,
                             show_progress_bar=False, convert_to_numpy=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
    rel = {d["id"]: {i for i, c in enumerate(chs) if d["gold_page"] in c["pdf_pages"]} for d in answerable}
    rows = []
    for d in answerable:
        q = models[key].encode([MODELS[key]["q_prefix"] + d["q"]], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        _, ids = index.search(q, TOP_K_MAX)
        ranked = [int(i) for i in ids[0] if i >= 0]
        for k in K_VALUES:
            rows.append({"lang": d["lang"], "k": k, **metrics_for(ranked, rel[d["id"]], k)})
    df = pd.DataFrame(rows)
    return df.groupby("k")[["hit", "precision", "mrr"]].mean().round(3)

files = sorted(EXP_DIR.glob("chunks_*.jsonl"), key=lambda p: int(p.stem.split("_")[1])) if EXP_DIR.exists() else []
if not files:
    print("No chunk-size files found. Run Cell W2-6b in the Week 2 notebook first.")
else:
    size_rows = []
    for p in files:
        size = int(p.stem.split("_")[1])
        res = evaluate_chunk_set(BEST_DENSE, read_jsonl(p))
        for k in K_VALUES:
            size_rows.append({"max_chars": size, "k": k, "chunks": len(read_jsonl(p)), **res.loc[k].to_dict()})
    size_df = pd.DataFrame(size_rows)
    print("Dense model used:", BEST_DENSE)
    display(size_df.pivot_table(index=["max_chars", "chunks"], columns="k", values="hit").rename(columns=lambda k: f"hit@{k}"))

Dense model used: bge_m3


,k,hit@1,hit@3,hit@5
max_chars,chunks,,,
600,2474,0.833,1.000,1.000
800,1664,0.778,0.944,0.944
1000,1281,0.833,0.944,1.000
1500,801,0.722,0.889,0.889


## Cell 11: Save results and write the summary

In [12]:
results_df = pd.concat(all_results, ignore_index=True)
results_path = LOGS / "week3_results.csv"
results_df.to_csv(results_path, index=False)
print("Saved per-question results to", results_path)

with open(INDEX_DIR / "dev_questions.json", "w", encoding="utf-8") as f:
    json.dump(DEV_QUESTIONS, f, ensure_ascii=False, indent=1)

!pip freeze | grep -iE "^(pypdf|pymupdf|pdfplumber|sentence-transformers|faiss-cpu|rank-bm25)==" > /content/drive/MyDrive/AI3-Project/requirements.txt
!cat /content/drive/MyDrive/AI3-Project/requirements.txt

Saved per-question results to /content/drive/MyDrive/AI3-Project/logs/week3_results.csv
faiss-cpu==1.16.0
rank-bm25==0.2.2
sentence-transformers==5.7.0


In [13]:
def _row(system, lang, k):
    r = compare[(compare["system"] == system) & (compare["lang"] == lang) & (compare["k"] == k)]
    return r.iloc[0] if len(r) else None

lines = ["| system | English hit@1 / @3 / @5 | Tagalog hit@1 / @3 / @5 | all MRR@5 |", "|---|---|---|---|"]
for system in compare["system"].unique():
    en = " / ".join(f"{_row(system, 'en', k)['hit']:.2f}" for k in K_VALUES)
    tl = " / ".join(f"{_row(system, 'tl', k)['hit']:.2f}" for k in K_VALUES)
    lines.append(f"| {system} | {en} | {tl} | {_row(system, 'all', TOP_K_MAX)['mrr']:.2f} |")

n_en = sum(d["lang"] == "en" for d in answerable)
n_tl = sum(d["lang"] == "tl" for d in answerable)
display(Markdown(f'''## Week 3 summary
- **Index:** {len(chunks)} chunks (max {max(c["char_len"] for c in chunks)} characters), exact FAISS `IndexFlatIP` per model, saved with the chunk metadata to `{INDEX_DIR.name}/` on Drive.
- **Questions:** {len(DEV_QUESTIONS)} dev questions ({len(answerable)} answerable: {n_en} English, {n_tl} Tagalog/Taglish; {len(unanswerable)} unanswerable). Relevant = the chunk's `pdf_pages` contains the gold page.
- **Best system on these dev questions:** `{BEST}`.

{chr(10).join(lines)}

**Limits to state in the report:** the dev set is small ({len(answerable)} answerable questions), so one question moves hit@k by about {100 / len(answerable):.0f} points and the Tagalog numbers rest on only {n_tl} questions. Chunk size 1000 stays provisional until the held-out test questions confirm it. Recall@k is low by design because a page is split over several chunks.
'''))

## Week 3 summary
- **Index:** 1281 chunks (max 1000 characters), exact FAISS `IndexFlatIP` per model, saved with the chunk metadata to `index/` on Drive.
- **Questions:** 20 dev questions (18 answerable: 14 English, 4 Tagalog/Taglish; 2 unanswerable). Relevant = the chunk's `pdf_pages` contains the gold page.
- **Best system on these dev questions:** `dense:bge_m3`.

| system | English hit@1 / @3 / @5 | Tagalog hit@1 / @3 / @5 | all MRR@5 |
|---|---|---|---|
| bm25 | 0.57 / 0.71 / 0.79 | 0.75 / 0.75 / 0.75 | 0.68 |
| dense:bge_m3 | 0.79 / 0.93 / 1.00 | 1.00 / 1.00 / 1.00 | 0.90 |
| dense:e5_base | 0.57 / 0.86 / 0.93 | 1.00 / 1.00 / 1.00 | 0.79 |
| dense:minilm | 0.50 / 0.71 / 0.79 | 0.75 / 0.75 / 1.00 | 0.66 |
| hybrid:bge_m3 | 0.71 / 1.00 / 1.00 | 1.00 / 1.00 / 1.00 | 0.89 |
| hybrid:e5_base | 0.64 / 0.93 / 0.93 | 1.00 / 1.00 / 1.00 | 0.82 |
| hybrid:minilm | 0.57 / 0.79 / 0.93 | 1.00 / 1.00 / 1.00 | 0.76 |

**Limits to state in the report:** the dev set is small (18 answerable questions), so one question moves hit@k by about 6 points and the Tagalog numbers rest on only 4 questions. Chunk size 1000 stays provisional until the held-out test questions confirm it. Recall@k is low by design because a page is split over several chunks.
